# Week 4: Pre-Class Practice - Cross-Validation & Regularization Basics

**Goal:** Practice cross-validation and regularization basics before the live session

**Dataset:** Breast Cancer (569 samples, 30 features)

**Time:** 15-20 minutes

---

## What You'll Practice

- Using `cross_val_score()` instead of single train/test split
- Comparing single split vs k-fold CV results
- Training models with regularization (Ridge vs Lasso)
- Understanding alpha parameter effect

**Scaffolding: 70% provided** - You'll fill in CV and regularization function calls

---

## Step 1: Imports

All imports are provided. Just run this cell.

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt

# For Jupyter notebooks
%matplotlib inline

print("✅ All imports successful!")

## Step 2: Load Breast Cancer Dataset

This is a classic binary classification dataset: predicting malignant vs benign tumors.

In [ ]:
# Load Breast Cancer dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

print(f"Dataset shape: {X.shape}")
print(f"Number of classes: {len(np.unique(y))}")
print(f"Class distribution: {np.bincount(y)}")
print(f"Target names: {cancer.target_names}")

## Step 3: Scale Features

Scaling is important for logistic regression and regularized models.

In [ ]:
# Scale features to mean=0, std=1
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"✅ Features scaled")
print(f"Mean of first feature: {X_scaled[:, 0].mean():.6f} (should be ~0)")
print(f"Std of first feature: {X_scaled[:, 0].std():.6f} (should be ~1)")

## Step 4: Single Train/Test Split (Old Way)

First, let's do what we did in Week 3 - a single train/test split.

In [ ]:
# Single split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

# Train logistic regression
model_single = LogisticRegression(random_state=42, max_iter=10000)
model_single.fit(X_train, y_train)

# Evaluate
y_pred = model_single.predict(X_test)
single_accuracy = accuracy_score(y_test, y_pred)

print(f"Single Split Accuracy: {single_accuracy:.3f}")

### Problem: Is 0.XXX the "true" accuracy?

Let's check by trying different random_state values:

In [ ]:
# Try 10 different random splits
single_split_scores = []

for i in range(10):
    X_train, X_test, y_train, y_test = train_test_split(
        X_scaled, y, test_size=0.2, random_state=i
    )
    model = LogisticRegression(random_state=42, max_iter=10000)
    model.fit(X_train, y_train)
    score = model.score(X_test, y_test)
    single_split_scores.append(score)
    print(f"Split {i+1}: {score:.3f}")

print(f"\nRange: {min(single_split_scores):.3f} to {max(single_split_scores):.3f}")
print(f"Variability: {max(single_split_scores) - min(single_split_scores):.3f}")
print(f"Mean: {np.mean(single_split_scores):.3f}")
print(f"Std Dev: {np.std(single_split_scores):.3f}")

**Notice:** Different splits give different scores! Which one is "correct"? We don't know!

---

## YOUR TURN: Cross-Validation (Better Way)

Instead of one split, let's use k-fold cross-validation to get a more reliable estimate.

In [ ]:
# TODO: Perform 5-fold cross-validation
# Hint: Use cross_val_score(model, X_scaled, y, cv=5)
# Expected: Array of 5 accuracy scores

model_cv = LogisticRegression(random_state=42, max_iter=10000)

cv_scores = cross_val_score(______, ______, ______, cv=______)

print("Cross-Validation Scores (5 folds):")
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i}: {score:.3f}")

# TODO: Calculate mean and standard deviation
# Hint: Use cv_scores.mean() and cv_scores.std()
# Expected: Single mean value and std value

cv_mean = ______
cv_std = ______

print(f"\n📊 CV Results: {cv_mean:.3f} ± {cv_std:.3f}")
print(f"\n✅ Cross-validation complete!")

### Compare: Single Split vs Cross-Validation

In [ ]:
print("=" * 60)
print("COMPARISON: Single Split vs Cross-Validation")
print("=" * 60)
print(f"\nSingle Split (random_state=42): {single_accuracy:.3f}")
print(f"  Issue: Different random_state gives {min(single_split_scores):.3f} to {max(single_split_scores):.3f}")
print(f"\n5-Fold CV: {cv_mean:.3f} ± {cv_std:.3f}")
print(f"  Benefit: More reliable - averages across 5 different splits")
print("=" * 60)

---

## YOUR TURN: Regularization Comparison

Now let's compare Ridge (L2) vs Lasso (L1) regularization.

**Context:** Regularization adds a penalty to prevent overfitting.

In [ ]:
# We'll use Ridge for regression (same concept as regularized LogisticRegression)
# Let's create a regression version of this problem
from sklearn.datasets import load_diabetes

# Load diabetes dataset (regression problem)
diabetes = load_diabetes()
X_diab = diabetes.data
y_diab = diabetes.target

# Scale features
X_diab_scaled = StandardScaler().fit_transform(X_diab)

print(f"Diabetes dataset: {X_diab.shape}")
print("Task: Predict disease progression using 10 features")

### Test Ridge Regression with Different Alpha Values

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import cross_val_score

# Test different alpha values
alphas = [0.01, 0.1, 1.0, 10.0, 100.0]

ridge_scores = []
lasso_scores = []

for alpha in alphas:
    # TODO: Create Ridge model with this alpha
    # Hint: Ridge(alpha=alpha, random_state=42)
    # Expected: Ridge model object
    
    ridge_model = ______
    
    # TODO: Create Lasso model with this alpha
    # Hint: Lasso(alpha=alpha, random_state=42, max_iter=10000)
    # Expected: Lasso model object
    
    lasso_model = ______
    
    # Evaluate with cross-validation (R² score for regression)
    ridge_cv = cross_val_score(ridge_model, X_diab_scaled, y_diab, cv=5, scoring='r2')
    lasso_cv = cross_val_score(lasso_model, X_diab_scaled, y_diab, cv=5, scoring='r2')
    
    ridge_scores.append(ridge_cv.mean())
    lasso_scores.append(lasso_cv.mean())
    
    print(f"Alpha = {alpha:6.2f}  |  Ridge: {ridge_cv.mean():.3f}  |  Lasso: {lasso_cv.mean():.3f}")

print("\n✅ Regularization comparison complete!")

### Visualize Regularization Effect

In [ ]:
# Plot Ridge vs Lasso performance across alpha values
plt.figure(figsize=(10, 6))
plt.plot(alphas, ridge_scores, marker='o', label='Ridge (L2)', linewidth=2)
plt.plot(alphas, lasso_scores, marker='s', label='Lasso (L1)', linewidth=2)
plt.xscale('log')
plt.xlabel('Alpha (Regularization Strength)', fontsize=12)
plt.ylabel('Cross-Validation R² Score', fontsize=12)
plt.title('Ridge vs Lasso: Effect of Regularization Strength', fontsize=14)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.axhline(y=0, color='red', linestyle='--', alpha=0.3, label='Baseline')
plt.show()

# Find best alpha for each
best_ridge_idx = np.argmax(ridge_scores)
best_lasso_idx = np.argmax(lasso_scores)

print(f"\nBest Ridge alpha: {alphas[best_ridge_idx]} (R² = {ridge_scores[best_ridge_idx]:.3f})")
print(f"Best Lasso alpha: {alphas[best_lasso_idx]} (R² = {lasso_scores[best_lasso_idx]:.3f})")

**Key Observations:**
- **Small alpha:** Less regularization, model more complex (might overfit)
- **Large alpha:** More regularization, model simpler (might underfit)
- **Sweet spot:** Somewhere in the middle (we'll use GridSearchCV to find this automatically!)

---

## Summary: What You've Practiced

✅ **Cross-Validation:**
- Single train/test split can be misleading (varies by random_state)
- k-fold CV gives more reliable performance estimates
- Report as mean ± std dev (e.g., "0.847 ± 0.012")

✅ **Regularization:**
- Ridge (L2) and Lasso (L1) add penalties to prevent overfitting
- Alpha controls regularization strength
- Need to try different alphas to find best one (GridSearchCV automates this!)

---

## 🎉 Congratulations!

You've practiced:
- ✅ Using `cross_val_score()` for reliable performance estimates
- ✅ Understanding why single splits are unreliable
- ✅ Comparing Ridge vs Lasso regularization
- ✅ Seeing how alpha affects model performance

**Questions to think about:**
1. Why did different random_state values give different accuracy scores?
2. What's better about CV results compared to single split?
3. What happens when alpha is very small vs very large?

**Ready for the live session!** You'll learn:
- GridSearchCV (automates hyperparameter tuning)
- Data leakage and how Pipeline prevents it
- Building complete production pipelines

---

*Next: Complete Week 4 live session for full production ML workflow*